# Lecture 6 — Spatial Heterogeneity and Calibration

**Computational Methods for Heterogeneous-Agent Macro**

Jeffrey Sun


### Environment

Activate the project, load `HouseholdStages` plus `Printf` / `Plots`.


In [ ]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()
using HouseholdStages
using Printf
using Plots

## 1 · The spatial setup

Three locations `loc1`, `loc2`, `loc3` with productivities $A_1 > A_2 > A_3$. Each location produces a single, costlessly-tradable, perfectly-substitutable numeraire good with Cobb–Douglas technology
$$Y_j = A_j K_j^{\alpha} L_j^{1-\alpha}.$$
Capital flows freely across locations and with the rest of the world, so the country is small in the world bond market: **$r$ is exogenous**. Within the country, MPK equalization across locations pins $K_j / L_j$ given $(r, A_j)$, and wages then satisfy
$$w_j = (1-\alpha)\, A_j\, \big(\tfrac{\alpha A_j}{r+\delta}\big)^{\alpha/(1-\alpha)}.$$

**Link to L04.** We close the model differently than L04: $r$ is set by the world bond market, so no $K$-tatonnement is needed. The outer loop is the **calibration** of preference shifters $\alpha = (\alpha_1, \alpha_2, \alpha_3)$ — see §5 — which replaces market clearing as the equilibrium condition.

Households are heterogeneous in **wealth** and **location** only (no idiosyncratic income shocks). Within a period:

3. **Consumption

The chain is
$$\text{Migration} \circ \text{WealthChange} \circ \text{ConsumptionSavings}.$$


### Parameters and the target shares

Standard macro calibration. The world rate is $r = 0.03$. Productivities are $(1.20, 1.00, 0.85)$ — `loc1` is the high-wage region. Migration cost $C[i, j]$ is symmetric, with adjacent locations cheaper to move between than distant ones. The preference shifters start at $\alpha = (0, 0, 0)$ as an initial guess for the calibration loop — by the normalization $\alpha_1 \equiv 0$, only $(\alpha_2, \alpha_3)$ are free.

The target population shares `s_data = (0.30, 0.30, 0.40)` are synthetic: the *data* say 40% of households live in `loc3` (the lowest-productivity region), so we will need a positive preference shifter on `loc3` to rationalize them. The whole point of the calibration in §5 is to recover that shifter.


In [ ]:
@kwdef struct SpatialParams3
    β::Float64 = 0.96
    σ::Float64 = 1.5
    α::Float64 = 0.36
    δ::Float64 = 0.08
    r::Float64 = 0.03                          # world interest rate (exogenous)
    A::NTuple{3,Float64} = (1.20, 1.00, 0.85)  # productivities
    # Migration cost matrix C[i, j] (origin → destination).
    C_base::Matrix{Float64} = [0.0 0.5 1.0;
                               0.5 0.0 0.5;
                               1.0 0.5 0.0]
    η_logit::Float64 = 1.5                     # Gumbel scale
    # Initial guess for the (α₂, α₃) calibration iterate; α₁ ≡ 0.
    α_init::NTuple{2,Float64} = (0.0, 0.0)
    N_w::Int       = 250
    w_min::Float64 = 0.0
    w_max::Float64 = 25.0
end
Base.Broadcast.broadcastable(p::SpatialParams3) = Ref(p)

const LOC_NAMES = (:loc1, :loc2, :loc3)
loc_idx(sym::Symbol) = findfirst(==(sym), LOC_NAMES)

const s_data = [0.30, 0.30, 0.40]
p = SpatialParams3()
@printf "β=%.2f, σ=%.2f, α=%.2f, δ=%.2f, r=%.4f\n" p.β p.σ p.α p.δ p.r
@printf "A = (%.2f, %.2f, %.2f);  η_logit = %.2f\n" p.A[1] p.A[2] p.A[3] p.η_logit
println("target shares = ", s_data)

## 2 · Wages from the world rate

`spatial_wage(r, A_j, p)` implements $w_j = (1-\alpha) A_j (\alpha A_j / (r+\delta))^{\alpha/(1-\alpha)}$. We broadcast it over `p.A` to get the length-3 wage vector `w_vec`. The env we pass to the household block will carry `w_vec` and the current calibration iterate $\alpha$ (also length 3) — that comes once we've built the chain.


In [ ]:
function spatial_wage(r, A_j, p)
    return (1 - p.α) * A_j * (p.α * A_j / (r + p.δ))^(p.α / (1 - p.α))
end

w_vec = spatial_wage.(p.r, p.A, p)
@printf "Wages: w = (%.4f, %.4f, %.4f)\n" w_vec[1] w_vec[2] w_vec[3]
@printf "Ratio w[1]/w[3] = %.3f (productivity ratio A[1]/A[3] = %.3f)\n" w_vec[1]/w_vec[3] p.A[1]/p.A[3]

## 3 · The household block

Three stages. The migration stage carries an **amenity closure** that reads the preference-shifter vector $\alpha$ from `env`:
$$\text{amenity}(j;\, \texttt{env}) = \texttt{env.}\alpha[j],$$
with the convention $\alpha_1 \equiv 0$ pinning location utility (only differences are identified). The library materializes the length-3 amenity vector once per backward pass, so the hot path is no slower than a static-vector amenity.

$\alpha$ is data on the env, not on the Spec. The calibration loop in §5 will vary $\alpha$ by **rebuilding env** each iteration — no in-place Spec mutation.


In [ ]:
_u_crra(c, ::Val{1})           = log(c)
_u_crra(c, ::Val{σv}) where σv = (c^(1 - σv)) / (1 - σv)
u_crra(c, valσ::Val) = c < 0 ? -Inf : _u_crra(c, valσ)

function spatial_household(p::SpatialParams3)
    layout = StateLayout(
        StateAxis(:wealth, continuous_grid(p.w_min, p.w_max; length=p.N_w, spacing=:log)),
        StateAxis(:location, categorical(collect(LOC_NAMES))),
    )

    # Amenity closure: indexes env.α[j] at the destination location.
    amenity = (dest; env) -> env.α[loc_idx(dest)]

    migration = MigrationStage(layout; migration_cost=p.C_base, amenity, ε=p.η_logit)

    income = WealthChangeStage(layout; wealth_post = function (cell; env)
        return (1 + env.r) * cell.wealth + env.w[loc_idx(cell.location)]
    end)

    savings = ConsumptionSavingsStage(layout; β=p.β, utility=(cell, c; env) -> u_crra(c, Val(p.σ)), monotone_search=:divide_conquer)

    hh = migration ∘ income ∘ savings

    return define_moments!(hh;
        K_total = at_end(integrand=(cell; env) -> cell.wealth, reduce=sum),
        L       = at_end(integrand=(cell; env) -> Float64[cell.location == j for j in LOC_NAMES], reduce=sum),
    )
end

hh = spatial_household(p)
dims = layout_size(first(hh.spec.stages).input_layout)
@printf "Layout: wealth %d × location %d = %d cells\n" dims[1] dims[2] prod(dims)

### 3.1 What's inside `hh`?

A quick peek at the migration stage's Spec and Buffer. The Spec carries the cost matrix and the amenity *closure*; the Buffer carries the per-cell choice-probability tensor `choice_prob` — the migration stage's "memory" — repopulated by the log-sum-exp on every `backward!`.


In [ ]:
mig = hh.spec.stages[1]
mig_buf = hh.buffer.stages[1]
@show typeof(mig)
@show typeof(mig.amenity)                       # closure: (dest; env) -> Real
@show size(mig_buf.kernel.choice_prob)          # (N_w, 3, 3): per (wealth, origin), per destination

## 4 · Solving the household at $\alpha = 0$

A **demo run** at the initial guess $\alpha = 0$ — *before* the calibration loop kicks in. Choice probabilities depend only on wages and the migration cost. `loc1` has the highest wage; we expect it to attract the largest population share. The §5 calibration will then bend $\alpha$ to match the data.


In [ ]:
function solve_household(hh, p, α; V_init=nothing, Λ_init=nothing)
    env = make_env(hh; r=p.r, w=spatial_wage.(p.r, p.A, p), α)

    (;V, Λ, moments, history) = solve_steady_state_given_env!(hh, env; V_init, Λ_init)
    shares = moments.L ./ sum(moments.L)

    return (; V, Λ, env, moments, shares,
              vfi_iters=history.vfi_iters, lambda_iters=history.lambda_iters)
end

out0 = solve_household(hh, p, [0.0, 0.0, 0.0])
@printf "mass conservation: ΣΛ = %.10f\n" sum(out0.Λ)
@printf "VFI %d iters, Λ %d iters\n" out0.vfi_iters out0.lambda_iters
@printf "Population shares (uncalibrated): (%.3f, %.3f, %.3f)\n" out0.shares[1] out0.shares[2] out0.shares[3]
@printf "Total wealth K_total = %.4f\n" out0.moments.K_total

The high-productivity region `loc1` is overpopulated relative to the data; `loc3` is underpopulated. Something is keeping people in `loc3` that the model — with $\alpha = 0$ — does not see. That something is what we will calibrate.


In [ ]:
xs = 1:3
plt_init = plot(title="Uncalibrated vs target shares", ylabel="share",
                xticks=(xs, ["loc1", "loc2", "loc3"]),
                ylims=(0.0, 0.5), legend=:topright, size=(640, 360))
bar!(plt_init, xs .- 0.18, out0.shares; bar_width=0.30, label="model (α=0)")
bar!(plt_init, xs .+ 0.18, s_data;      bar_width=0.30, label="target (data)")

## 5 · Calibration by damped log-share update

The wage-only model misses the data on `loc3`. We need preference shifters $(\alpha_2, \alpha_3)$ — recall $\alpha_1 \equiv 0$ — such that the stationary population shares equal `s_data`. Three steps: understand the map $\alpha \to \text{shares}$, write down its (approximate) inverse, then run the loop.


### 5.1 The map $\alpha \to$ shares

Intuition: a larger amenity $\alpha_j$ makes destination $j$ more attractive, so more households migrate there in steady state. Concretely, bumping $\alpha_3$ from 0 to +0.5 should shift mass *into* `loc3` and *out of* `loc1` / `loc2`. Let's check.


In [ ]:
out_bump = solve_household(hh, p, [0.0, 0.0, 0.5])
@printf "α₃ = 0.0:  shares = (%.3f, %.3f, %.3f)\n" out0.shares[1] out0.shares[2] out0.shares[3]
@printf "α₃ = 0.5:  shares = (%.3f, %.3f, %.3f)\n" out_bump.shares[1] out_bump.shares[2] out_bump.shares[3]
@printf "Δ on loc3: %+.3f\n" out_bump.shares[3] - out0.shares[3]

### 5.2 Inverting the map — a damped log-share update

In **static** logit, larger amenity $\alpha_j$ shifts mass toward $j$ proportionally to $\log s_j$. The update
$$\alpha_j \;\leftarrow\; \alpha_j \;+\; \eta_{\text{logit}} \cdot \bigl(\log s_j^{\text{data}} - \log s_j^{\text{model}}\bigr)$$
is a damped step toward the fixed point in the log-share gap. In our **dynamic** model, $V_j$ also depends on $\alpha$ through future migration value, so the static formula is no longer an exact inversion — but the same update remains a useful, empirically conservative, fixed-point iteration. **In code:** scale by `update_speed` $\in (0, 1]$ for safety, drop the gradient of $\alpha_1$ to enforce the normalization, and let it run.


In [ ]:
function calibrate_shifters!(hh, p, s_data; verbosity=1)
    update_speed = 0.1
    tol     = 5e-3
    maxiter = 60

    # α = (α₁, α₂, α₃) with α₁ ≡ 0 by normalization.
    α = Float64[0.0, p.α_init[1], p.α_init[2]]
    V, Λ = nothing, nothing
    α_history   = Vector{Float64}[copy(α)]
    gap_history = Vector{Float64}[]
    last_out    = nothing
    iters       = 0
    converged   = false

    while iters < maxiter
        out = solve_household(hh, p, α; V_init=V, Λ_init=Λ)
        last_out = out
        (;V, Λ) = out

        gap = log.(s_data) .- log.(max.(out.shares, 1e-8))
        push!(gap_history, copy(gap))
        iters += 1

        verbosity > 0 && @printf("  iter %2d: shares = (%.3f, %.3f, %.3f); α = (%.3f, %.3f, %.3f); ‖gap‖∞ = %.4f\n",
            iters, out.shares[1], out.shares[2], out.shares[3],
            α[1], α[2], α[3], maximum(abs.(gap)))

        if maximum(abs.(gap)) < tol
            converged = true
            break
        end

        # Damped log-share update: αⱼ ← αⱼ + s · η · (log sᵈᵃᵗᵃ − log sᵐᵒᵈᵉˡ); renormalize α[1] = 0.
        α .+= update_speed * p.η_logit .* gap
        α .-= α[1]
        push!(α_history, copy(α))
    end

    return (; α, iters, converged, α_history, gap_history,
              shares=last_out.shares, V=last_out.V, Λ=last_out.Λ,
              env=last_out.env, moments=last_out.moments)
end

### Run the calibration

The loop starts at $\alpha = 0$ (where `loc3` is underpopulated), takes one damped log-share step, re-solves the household, repeats. Convergence is geometric.


In [ ]:
calib = calibrate_shifters!(hh, p, s_data; verbosity=1)
@printf "\nFinal α = (%.4f, %.4f, %.4f); converged = %s in %d iterations\n" calib.α[1] calib.α[2] calib.α[3] calib.converged calib.iters
@printf "Final shares: (%.3f, %.3f, %.3f) vs target (%.3f, %.3f, %.3f)\n" calib.shares[1] calib.shares[2] calib.shares[3] s_data[1] s_data[2] s_data[3]

### Convergence diagnostics

The log-share gap collapses geometrically; the shifter trajectory rises monotonically toward the fixed point.


In [ ]:
αs   = hcat(calib.α_history...)
gaps = hcat(calib.gap_history...)

plt_α   = plot(1:size(αs, 2), αs',
               label=["α[1]" "α[2]" "α[3]"], marker=:circle, linewidth=2,
               xlabel="calibration iteration", ylabel="α_j",
               title="Preference shifter trajectory")

plt_gap = plot(1:size(gaps, 2), maximum.(abs, eachcol(gaps)),
               marker=:circle, linewidth=2, yaxis=:log,
               xlabel="calibration iteration", ylabel="max gap (log scale)",
               title="‖log s_data − log s_model‖∞", legend=false)

plot(plt_α, plt_gap; layout=(1, 2), size=(900, 360))

### Final shares: uncalibrated, calibrated, target


In [ ]:
xs = 1:3
plt_final = plot(title="Population shares", ylabel="share",
                 xticks=(xs, ["loc1", "loc2", "loc3"]),
                 ylims=(0.0, 0.5), legend=:topright, size=(640, 360))
bar!(plt_final, xs .- 0.25, out0.shares;  bar_width=0.22, label="uncalibrated (α=0)")
bar!(plt_final, xs,         calib.shares; bar_width=0.22, label="calibrated")
bar!(plt_final, xs .+ 0.25, s_data;       bar_width=0.22, label="target (data)")

## 6 · Reading the calibrated shifters

- $\alpha_1 = 0$ by normalization. `loc1`'s 30% share in the data is *fully explained* by its productivity advantage; it needs no amenity boost.
- $\alpha_2 \approx 0$ (about $-0.013$). `loc2`'s 30% share is also broadly consistent with the wage-only model.
- $\alpha_3 \approx 0.75$. `loc3` houses 40% of households despite having the *lowest* wage. The data are telling us about $0.75$ units of utility per period of "amenity" or unobserved preference that the wage model misses. This is the kind of inference indirect identification gives us: a value for a quantity we cannot see directly.


## 7 · A closer look: wealth distribution by location

§5 said $\alpha$ moves *mass* across locations. The dual question — given location, how is wealth distributed? — is where the wage differential shows up. With no income heterogeneity, the only buffer-stock motive is the *possibility* of future migration to a different wage, so we expect tight, location-conditional wealth distributions whose mean tracks the local wage.

Three summary statistics per location:


In [ ]:
"""CLAUDE
Compute the Gini coefficient of a discrete distribution `(values, mass)`
where `mass` need not sum to 1 (it's normalized internally). Values are
sorted ascending; the Lorenz-curve integral is evaluated by the trapezoid
rule on the sorted (cumulative-population, cumulative-wealth) pairs.
"""
function _gini(values::AbstractVector, mass::AbstractVector)
    total = sum(mass)
    total ≤ 0 && return NaN
    perm = sortperm(values)
    v    = values[perm]
    p    = mass[perm] ./ total
    cumpop    = vcat(0.0, cumsum(p))
    cumwealth = vcat(0.0, cumsum(v .* p) ./ sum(v .* p))
    area_under_lorenz = sum(diff(cumpop) .* (cumwealth[2:end] .+ cumwealth[1:end-1]) ./ 2)
    return 1 - 2 * area_under_lorenz
end

wgrid = first(hh.spec.stages).input_layout.axes[1].kind.grid
Λ_ss  = calib.Λ
w_vec = spatial_wage.(p.r, p.A, p)

println("location |  wage  |  share | mean w  | Pr(w=0) |  Gini")
println("---------|--------|--------|---------|---------|--------")
for j in 1:3
    mass_j  = @view Λ_ss[:, j]
    share_j = sum(mass_j)
    mean_w  = sum(wgrid .* mass_j) / share_j
    p_atbc  = mass_j[1] / share_j
    gini_j  = _gini(wgrid, mass_j)
    @printf "%-8s | %.4f | %.3f  | %.4f  |  %.3f  | %.3f\n" string(LOC_NAMES[j]) w_vec[j] share_j mean_w p_atbc gini_j
end

**Reading the table.** `loc1` is the high-wage region: by §5's calibration it houses only 30% of households, but each one is wealthier (highest mean $b$, smallest constrained mass). `loc3` carries 40% of the population at the *lowest* wage — the amenity $\alpha_3$ pulls households in, and because some of them have low savings buffers and the wage doesn't help them build one, the constrained mass is larger and the Gini is higher. The two channels — $\alpha$ moves *mass*, wage moves *wealth-given-location* — are visible separately in the same picture.


### 7.1 Conditional wealth distribution by location

The marginals $\Lambda[:, j]$ scale with the *population* of location $j$, which makes a naive plot misleading — `loc1`'s lower curve gets misread as "the location is poorer" when it is actually just less populated. We normalize each column to integrate to 1 and plot the *conditional* distribution of wealth given location. Wealth is on a log axis because the grid is log-spaced (dense near zero, coarse at the top).

Each curve integrates to 1 — these are directly comparable in *shape*, not in *level*.


In [ ]:
Λ_cond = Λ_ss ./ sum(Λ_ss; dims=1)   # normalize each location column to sum to 1
plot(wgrid, Λ_cond;
     labels=["loc1" "loc2" "loc3"],
     xlims=(wgrid[2], wgrid[end]),   # skip wgrid[1] = 0 for log scale
     xscale=:log10,
     xlabel="wealth (log scale)", ylabel="conditional mass",
     title="Wealth | location  (each curve integrates to 1)",
     linewidth=2, size=(720, 360))

## 8 · Foreshadow

The spatial model's bond market was exogenous: we set $r$ from the world. **L07** brings the asset market back inside the model and adds *aggregate* uncertainty — the Krusell–Smith problem.
